In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Load the merged dataset
df = pd.read_csv('/Users/yaswanthkanagarla/Desktop/Master_Thesis/BD_code/Battery_Diagnostics_Thesis/scripts/final_7features_state_IX.csv')

# Convert 'state' if needed
state_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6, 'VII': 7, 'VIII': 8, 'IX': 9}
df['state'] = df['state'].map(state_map)

# 🔍 Check which columns you’re using
#feature_cols = [col for col in df.columns if col not in ['SOH', 'capacity', 'cycle', 'cell', 'T_C','state', 'Zmag_1000Hz','R_sei', 'R_s','Zmag_100Hz','C_dl_est','Zmag_0.1Hz','Zmag_10Hz','R_ct','tail_slope','tail_angle_deg','f_peak_main','Phase_100Hz', 'Phase_0.1Hz','Phase_1000Hz','Phase_10Hz']]  # Adjust as needed
#target_col = 'SOH'

In [8]:
import numpy as np
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.metrics import mean_squared_error

# ---- pick your feature columns ----
# Option 1 (recommended): explicitly set them
feature_cols = [
    "R_s_rel","R_ct_rel","R_sei_rel","Zmag_0.1Hz_rel",
    "C_dl_log","tail_slope","Zmag_1000Hz_rel","T_C"
]

target_col = "SOH"

# ---- create group = (temp, cell) so no leakage across cell/temp ----
df["group_ct"] = df["T_C"].astype(str) + "_cell" + df["cell"].astype(str)

# keep only State IX (state == 9) if your CSV contains multiple states
df9 = df[df["state"] == 9].copy()

# clean inf/nan
df9[feature_cols] = df9[feature_cols].replace([np.inf, -np.inf], np.nan)
df9 = df9.dropna(subset=feature_cols + [target_col]).copy()

X = df9[feature_cols]
y = df9[target_col]
groups = df9["group_ct"]

def metrics(y_true, y_pred):
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

def make_rf():
    return RandomForestRegressor(
        n_estimators=800,
        random_state=42,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2
    )


groupkfolds

In [11]:
gkf = GroupKFold(n_splits=4)

rows = []
for fold, (tr, va) in enumerate(gkf.split(X, y, groups=groups), start=1):
    X_train, X_val = X.iloc[tr], X.iloc[va]
    y_train, y_val = y.iloc[tr], y.iloc[va]

    model = make_rf()
    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    m = metrics(y_val, pred)

    held_out = groups.iloc[va].unique().tolist()
    rows.append({
        "fold": fold,
        "n_val_groups": len(held_out),
        "val_groups": ", ".join(held_out),
        **m,
        "n_train": len(tr),
        "n_val": len(va)
    })

gkf_df = pd.DataFrame(rows)
print("GroupKFold (4 folds)")
print("R2   mean±std:", gkf_df["R2"].mean(), "±", gkf_df["R2"].std())
print("MAE  mean±std:", gkf_df["MAE"].mean(), "±", gkf_df["MAE"].std())
print("RMSE mean±std:", gkf_df["RMSE"].mean(), "±", gkf_df["RMSE"].std())

gkf_df.sort_values("R2")


GroupKFold (4 folds)
R2   mean±std: 0.7702347362030831 ± 0.07475276711210303
MAE  mean±std: 0.0607239364096101 ± 0.03038485561983779
RMSE mean±std: 0.0742749940088364 ± 0.03737541368481404


,fold,n_val_groups,val_groups,R2,MAE,RMSE,n_train,n_val
2,3,3,"25_cell8, 35_cell1, 45_cell1",0.689700,0.050733,0.063032,2074,662
1,2,3,"25_cell2, 25_cell5, 25_cell6",0.747367,0.088400,0.113313,2016,720
0,1,3,"25_cell1, 25_cell3, 25_cell7",0.774983,0.081441,0.093196,2045,691
3,4,3,"25_cell4, 35_cell2, 45_cell2",0.868889,0.022321,0.027559,2073,663


loco

In [4]:
logo = LeaveOneGroupOut()

rows = []
for fold, (tr, va) in enumerate(logo.split(X, y, groups=groups), start=1):
    X_train, X_val = X.iloc[tr], X.iloc[va]
    y_train, y_val = y.iloc[tr], y.iloc[va]

    model = make_rf()
    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    m = metrics(y_val, pred)

    held_out = groups.iloc[va].unique().tolist()
    rows.append({
        "fold": fold,
        "held_out_group": held_out[0] if len(held_out) == 1 else str(held_out),
        **m,
        "n_train": len(tr),
        "n_val": len(va)
    })

loco_df = pd.DataFrame(rows)
print("LOCO (Leave-One-(T_C,cell)-Out)")
print("R2   mean±std:", loco_df["R2"].mean(), "±", loco_df["R2"].std())
print("MAE  mean±std:", loco_df["MAE"].mean(), "±", loco_df["MAE"].std())
print("RMSE mean±std:", loco_df["RMSE"].mean(), "±", loco_df["RMSE"].std())

loco_df.sort_values("R2").head(10)


LOCO (Leave-One-(T_C,cell)-Out)
R2   mean±std: -0.724369941927082 ± 2.714751189520361
MAE  mean±std: 0.06405004162408105 ± 0.035118958329760225
RMSE mean±std: 0.07313952046968963 ± 0.03862975140569325


,fold,held_out_group,R2,MAE,RMSE,n_train,n_val
2,3,25_cell3,-7.817826,0.122271,0.123588,2534,202
3,4,25_cell4,-4.563905,0.063395,0.066860,2701,35
1,2,25_cell2,-1.237961,0.045124,0.049032,2555,181
7,8,25_cell8,-0.289317,0.048250,0.052859,2699,37
5,6,25_cell6,0.210229,0.061978,0.083883,2524,212
10,11,45_cell1,0.415106,0.043087,0.051793,2437,299
6,7,25_cell7,0.602867,0.074151,0.084398,2596,140
8,9,35_cell1,0.676020,0.058912,0.073141,2410,326
4,5,25_cell5,0.704792,0.139203,0.161869,2409,327
11,12,45_cell2,0.853144,0.022416,0.026432,2426,310
